In [66]:
#@title Cell 1
# ============================================
# Cell 1
# Install dependencies (Google Colab)
# ============================================

# OpenCV
!pip -q install opencv-python-headless

# Scientific libraries
!pip -q install scipy tqdm matplotlib numpy

# ffmpeg (used to composite overlay video)
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print("Environment ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Environment ready.


In [67]:
#@title Cell 2
# ============================================
# Verify installation
# ============================================
# ============================================
# Cell 2
# Imports
# ============================================
import scipy
import cv2
import numpy as np
import os
import csv
import subprocess

from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

print("\nAll packages loaded successfully")

OpenCV: 5.0.0
NumPy: 2.0.2
SciPy: 1.16.3

All packages loaded successfully


In [68]:
#@title Cell 3
# ============================================
# Cell 5
# Colour palette + ROI helper
# ============================================


# ---------------------------------------------------------------------------
# Colour palette
# ---------------------------------------------------------------------------

_PALETTE = [
    (255,  80,  80), (80,  255,  80), (80,   80, 255), (255, 255,  80),
    (255,  80, 255), (80,  255, 255), (255, 160,  80), (160, 255,  80),
    (80,  160, 255), (255,  80, 160), (160,  80, 255), (80,  255, 160),
    (200, 200,  80), (200,  80, 200), (80,  200, 200), (255, 140,  40),
    (40,  255, 140), (140,  40, 255), (255,  40, 140), (40,  140, 255),
]


def cell_color(track_id):
    return _PALETTE[track_id % len(_PALETTE)]



def make_circle_mask(height, width, cx, cy, radius):
    """
    Build a uint8 mask that is 255 inside the circle and 0 outside.
    Applied to the foreground difference image before thresholding so
    detections outside the well boundary are suppressed.
    """

    mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    cv2.circle(
        mask,
        (int(cx), int(cy)),
        int(radius),
        255,
        thickness=-1
    )

    return mask

In [69]:
#@title Cell 4
# ============================================
# Cell 6
# Watershed segmentation
# ============================================


def watershed_segment(mask, peak_min_dist=15):
    """
    Standard distance-transform watershed with auto-detected seeds.
    Returns (markers int32, n_seeds int).
    """

    dist = cv2.distanceTransform(
        mask,
        cv2.DIST_L2,
        5
    )

    dist_norm = cv2.normalize(
        dist,
        None,
        0,
        1.0,
        cv2.NORM_MINMAX
    )


    ksize = 2 * peak_min_dist + 1

    kernel_peak = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (ksize, ksize)
    )


    dist_dilated = cv2.dilate(
        dist_norm,
        kernel_peak
    )


    local_max = (
        (dist_norm == dist_dilated) &
        (dist_norm > 0)
    ).astype(np.uint8)


    n_seeds, seed_labels = cv2.connectedComponents(
        local_max
    )


    if n_seeds <= 1:
        return np.zeros_like(
            mask,
            dtype=np.int32
        ), 0



    dist_u8 = (
        dist_norm * 255
    ).astype(np.uint8)


    dist_bgr = cv2.cvtColor(
        dist_u8,
        cv2.COLOR_GRAY2BGR
    )


    markers = seed_labels.astype(
        np.int32
    )


    background_label = n_seeds

    markers[mask == 0] = background_label


    cv2.watershed(
        dist_bgr,
        markers
    )


    return markers, n_seeds - 1





def watershed_seeded(mask, seed_xys, height, width):
    """
    Watershed seeded from explicit (x, y) positions — used for merge splitting.

    Each seed_xy gets a unique label. The watershed floods outward from those
    exact points, splitting a merged blob back into one region per seed.

    Args:
        mask      : uint8 binary mask of the merged blob (255 = foreground)
        seed_xys  : list of (x, y) centroid positions to use as seeds
        height, width : frame dimensions

    Returns:
        markers   : int32 label map (1..n_seeds = cell regions)
        n_seeds   : number of seeds used
    """

    dist = cv2.distanceTransform(
        mask,
        cv2.DIST_L2,
        5
    )


    dist_norm = cv2.normalize(
        dist,
        None,
        0,
        1.0,
        cv2.NORM_MINMAX
    )


    markers = np.zeros(
        (height, width),
        dtype=np.int32
    )


    background_label = len(seed_xys) + 1


    for i, (sx, sy) in enumerate(seed_xys):

        sx = int(
            np.clip(
                sx,
                0,
                width - 1
            )
        )

        sy = int(
            np.clip(
                sy,
                0,
                height - 1
            )
        )


        markers[sy, sx] = i + 1



    markers[mask == 0] = background_label


    dist_u8 = (
        dist_norm * 255
    ).astype(np.uint8)


    dist_bgr = cv2.cvtColor(
        dist_u8,
        cv2.COLOR_GRAY2BGR
    )


    cv2.watershed(
        dist_bgr,
        markers
    )


    return markers, len(seed_xys)

def bernsen_threshold(image, window_size=31, contrast_threshold=15):
    """
    Bernsen local thresholding.

    Args:
        image:
            uint8 grayscale image
        window_size:
            local neighborhood size (must be odd)
        contrast_threshold:
            minimum local contrast required for foreground

    Returns:
        uint8 binary mask (0/255)
    """

    if window_size % 2 == 0:
        window_size += 1

    # Local min/max
    local_min = cv2.erode(
        image,
        np.ones((window_size, window_size), np.uint8)
    )

    local_max = cv2.dilate(
        image,
        np.ones((window_size, window_size), np.uint8)
    )

    # Bernsen threshold
    local_thresh = (
        local_min.astype(np.float32) +
        local_max.astype(np.float32)
    ) / 2

    # Local contrast
    local_contrast = (
        local_max.astype(np.float32) -
        local_min.astype(np.float32)
    )

    # Threshold comparison
    mask = (image > local_thresh).astype(np.uint8) * 255

    # Remove low contrast regions
    mask[local_contrast < contrast_threshold] = 0

    return mask

def otsu_threshold(image):
    """
    Global Otsu thresholding.
    Args:
        image:
            uint8 grayscale image

    Returns:
        uint8 binary mask (0/255)
    """

    _, mask = cv2.threshold(image,0,255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return mask

def apply_threshold(image, method="otsu",
                    bernsen_window=31,
                    bernsen_contrast=15):
    """
    Apply selected thresholding method.

    Args:
        image:
            uint8 grayscale image

        method:
            "otsu" or "bernsen"

    Returns:
        uint8 binary mask
    """

    method = method.lower()

    if method == "otsu":
        return otsu_threshold(image)

    elif method == "bernsen":
        return bernsen_threshold(
            image,
            window_size=bernsen_window,
            contrast_threshold=bernsen_contrast
        )

    else:
        raise ValueError(
            f"Unknown threshold method '{method}'. "
            "Choose 'otsu' or 'bernsen'."
        )

In [70]:
#@title Cell 5
# ============================================
# Cell 7
# Contour helpers
# ============================================


def contour_centroid(contour):

    M = cv2.moments(
        contour
    )

    if M["m00"] == 0:
        return None

    return (
        M["m10"] / M["m00"],
        M["m01"] / M["m00"]
    )



def interp_contour(c0, c1, t):
    """
    Linearly interpolate between two contours at parameter t in [0, 1].

    Both contours are resampled to the same number of points before
    interpolation so the blend is point-wise.

    Args:
        c0, c1 : contour arrays of shape (N, 1, 2)
        t      : interpolation parameter (0 = c0, 1 = c1)

    Returns:
        contour array of shape (N, 1, 2) as int32
    """

    n = max(
        len(c0),
        len(c1)
    )


    def resample(c, n):

        pts = c.reshape(
            -1,
            2
        ).astype(
            np.float32
        )


        diffs = np.diff(
            pts,
            axis=0
        )


        dists = np.concatenate(
            [
                [0],
                np.linalg.norm(
                    diffs,
                    axis=1
                ).cumsum()
            ]
        )


        total = dists[-1]


        if total == 0:

            return np.tile(
                pts[0],
                (n, 1)
            )


        new_d = np.linspace(
            0,
            total,
            n
        )


        xs = np.interp(
            new_d,
            dists,
            pts[:,0]
        )

        ys = np.interp(
            new_d,
            dists,
            pts[:,1]
        )


        return np.stack(
            [
                xs,
                ys
            ],
            axis=1
        )



    p0 = resample(
        c0,
        n
    )

    p1 = resample(
        c1,
        n
    )


    blended = (
        (1-t)*p0 +
        t*p1
    ).astype(
        np.int32
    )


    return blended.reshape(
        -1,
        1,
        2
    )

In [71]:
#@title Cell 6
# ============================================
# Cell 8
# Pass 1b — proximity-based track assignment
# ============================================


def assign_tracks(all_detections, max_dist=50):
    """
    Walk forward through per-frame detection lists and assign each detection
    to the nearest active track using the Hungarian algorithm.

    Args:
        all_detections : list of lists — all_detections[f] is a list of dicts,
                         each with keys "centroid" (x,y) and "contour".
        max_dist       : maximum centroid distance (px) for a valid match.

    Returns:
        tracks : dict mapping track_id → list of
                 {"frame": int, "centroid": (x,y), "contour": ndarray}

        frame_tracks : list of lists — frame_tracks[f] contains all detections
                       in frame f after assignment.
    """

    next_id = 0

    active = {}

    tracks = {}

    frame_tracks = [
        []
        for _ in all_detections
    ]


    for f, detections in enumerate(all_detections):

        if not detections:
            continue



        det_xys = np.array(
            [
                d["centroid"]
                for d in detections
            ],
            dtype=float
        )



        if not active:

            for d in detections:

                tid = next_id
                next_id += 1


                active[tid] = np.array(
                    d["centroid"]
                )


                tracks[tid] = []


                tracks[tid].append(
                    {
                        "frame": f,
                        "centroid": d["centroid"],
                        "contour": d["contour"]
                    }
                )


                frame_tracks[f].append(
                    {
                        "track_id": tid,
                        "centroid": d["centroid"],
                        "contour": d["contour"]
                    }
                )


            continue



        track_ids = list(
            active.keys()
        )


        track_xys = np.array(
            [
                active[tid]
                for tid in track_ids
            ],
            dtype=float
        )


        diff = (
            track_xys[:,None,:]
            -
            det_xys[None,:,:]
        )


        costs = np.linalg.norm(
            diff,
            axis=2
        )


        rows, cols = linear_sum_assignment(
            costs
        )


        matched_tracks = set()
        matched_dets = set()



        for r, c in zip(rows, cols):

            if costs[r,c] <= max_dist:

                tid = track_ids[r]


                active[tid] = det_xys[c]


                tracks[tid].append(
                    {
                        "frame": f,
                        "centroid": detections[c]["centroid"],
                        "contour": detections[c]["contour"]
                    }
                )


                frame_tracks[f].append(
                    {
                        "track_id": tid,
                        "centroid": detections[c]["centroid"],
                        "contour": detections[c]["contour"]
                    }
                )


                matched_tracks.add(r)

                matched_dets.add(c)



        # Birth new tracks
        for c, d in enumerate(detections):

            if c not in matched_dets:

                tid = next_id
                next_id += 1


                active[tid] = np.array(
                    d["centroid"]
                )


                tracks[tid] = [
                    {
                        "frame": f,
                        "centroid": d["centroid"],
                        "contour": d["contour"]
                    }
                ]


                frame_tracks[f].append(
                    {
                        "track_id": tid,
                        "centroid": d["centroid"],
                        "contour": d["contour"]
                    }
                )



        # Remove inactive tracks
        unmatched_track_ids = {
            track_ids[r]
            for r in range(len(track_ids))
            if r not in matched_tracks
        }


        for tid in unmatched_track_ids:

            del active[tid]



    return tracks, frame_tracks

In [72]:
#@title Cell 7
# ============================================
# Cell 9
# Pass 2 — bidirectional correction sweep
# Part 1/2
# ============================================


def correction_sweep(tracks, frame_tracks, all_masks, n_frames,
                     height, width,
                     max_gap=10, min_track_len=3,
                     merge_area_thresh=1.8):
    """
    Bidirectional correction sweep over the full track history.

    Applies:
        (a) Noise removal
        (b) Gap filling
        (c) Merge splitting

    Returns:
        corrected_frame_tracks, global_median_area
    """


    corrected = [
        dict()
        for _ in range(n_frames)
    ]


    for f, entries in enumerate(frame_tracks):

        for e in entries:

            corrected[f][
                e["track_id"]
            ] = e



    # ----------------------------------------
    # Noise removal
    # ----------------------------------------

    short_ids = {
        tid
        for tid, hist in tracks.items()
        if len(hist) < min_track_len
    }



    for f in range(n_frames):

        for tid in short_ids:

            corrected[f].pop(
                tid,
                None
            )



    for tid in short_ids:

        del tracks[tid]



    print(
        f"  Noise removal: dropped {len(short_ids)} short tracks."
    )



    # ----------------------------------------
    # Median area calculation
    # ----------------------------------------

    def track_median_area(hist):

        areas=[]

        for e in hist:

            c=e["contour"]

            if c is not None:

                areas.append(
                    cv2.contourArea(c)
                )


        return (
            float(np.median(areas))
            if areas
            else 0.0
        )



    track_areas = {
        tid: track_median_area(hist)
        for tid,hist in tracks.items()
    }



    all_areas = [
        a
        for a in track_areas.values()
        if a>0
    ]


    global_median_area = (
        float(np.median(all_areas))
        if all_areas
        else 1.0
    )



    gaps_filled = 0
    merges_fixed = 0



    # ----------------------------------------
    # Process each track
    # ----------------------------------------

    for tid, hist in tqdm(
        tracks.items(),
        desc="Correction sweep"
    ):


        hist_sorted = sorted(
            hist,
            key=lambda e:e["frame"]
        )


        med_area = (
            track_areas.get(
                tid,
                global_median_area
            )
            or global_median_area
        )



        for i in range(
            len(hist_sorted)-1
        ):


            e0 = hist_sorted[i]
            e1 = hist_sorted[i+1]


            f0 = e0["frame"]
            f1 = e1["frame"]


            gap = f1-f0-1



            if gap == 0:
                continue


            if gap > max_gap:
                continue



            xy0=np.array(
                e0["centroid"],
                dtype=float
            )


            xy1=np.array(
                e1["centroid"],
                dtype=float
            )


            c0=e0["contour"]
            c1=e1["contour"]



            for step in range(
                1,
                gap+1
            ):

                f_fill=f0+step


                t=step/(gap+1)


                cx_f = (
                    xy0[0]*(1-t)
                    +
                    xy1[0]*t
                )


                cy_f = (
                    xy0[1]*(1-t)
                    +
                    xy1[1]*t
                )
                                # ----------------------------------------
                # Check merge candidate
                # ----------------------------------------

                mask_f = all_masks[f_fill]

                blob_area = 0
                label_map = None
                blob_id = None


                if mask_f is not None:

                    ix = int(
                        np.clip(
                            cx_f,
                            0,
                            width-1
                        )
                    )

                    iy = int(
                        np.clip(
                            cy_f,
                            0,
                            height-1
                        )
                    )


                    if mask_f[iy,ix] > 0:

                        n_labels, label_map = cv2.connectedComponents(
                            mask_f
                        )


                        blob_id = label_map[
                            iy,
                            ix
                        ]


                        blob_area = int(
                            (label_map == blob_id).sum()
                        )



                is_merge = (
                    blob_area >
                    merge_area_thresh * med_area
                    and
                    blob_area >
                    merge_area_thresh * global_median_area
                )



                if is_merge:

                    blob_mask=np.zeros(
                        (height,width),
                        dtype=np.uint8
                    )


                    blob_mask[
                        label_map == blob_id
                    ] = 255



                    seed_xys=[]
                    seed_tids=[]



                    # Search all tracks crossing this frame

                    for other_tid, other_hist in tracks.items():

                        other_sorted = sorted(
                            other_hist,
                            key=lambda e:e["frame"]
                        )


                        before=[
                            e for e in other_sorted
                            if e["frame"] <= f_fill
                        ]


                        after=[
                            e for e in other_sorted
                            if e["frame"] >= f_fill
                        ]


                        if not before or not after:
                            continue



                        eb=before[-1]
                        ea=after[0]



                        if eb["frame"] == ea["frame"]:

                            pred_xy=np.array(
                                eb["centroid"]
                            )

                        else:

                            tt=(
                                f_fill-eb["frame"]
                            ) / (
                                ea["frame"]-eb["frame"]
                            )


                            pred_xy = (
                                np.array(
                                    eb["centroid"]
                                )*(1-tt)
                                +
                                np.array(
                                    ea["centroid"]
                                )*tt
                            )



                        px=int(
                            np.clip(
                                pred_xy[0],
                                0,
                                width-1
                            )
                        )


                        py=int(
                            np.clip(
                                pred_xy[1],
                                0,
                                height-1
                            )
                        )



                        if blob_mask[py,px] > 0:

                            seed_xys.append(
                                pred_xy
                            )

                            seed_tids.append(
                                other_tid
                            )



                    if len(seed_xys)>=2:

                        ws_markers,_ = watershed_seeded(
                            blob_mask,
                            seed_xys,
                            height,
                            width
                        )


                        for s_idx,s_tid in enumerate(seed_tids):

                            region=np.zeros(
                                (height,width),
                                dtype=np.uint8
                            )


                            region[
                                ws_markers == s_idx+1
                            ]=255



                            sub_contours,_=cv2.findContours(
                                region,
                                cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE
                            )


                            if not sub_contours:
                                continue



                            sub_c=max(
                                sub_contours,
                                key=cv2.contourArea
                            )


                            sub_M=cv2.moments(
                                sub_c
                            )


                            if sub_M["m00"]==0:
                                continue



                            scx=sub_M["m10"]/sub_M["m00"]
                            scy=sub_M["m01"]/sub_M["m00"]



                            corrected[f_fill][s_tid]={
                                "track_id":s_tid,
                                "centroid":(scx,scy),
                                "contour":sub_c,
                                "corrected":"merge_split"
                            }



                        merges_fixed+=1

                        continue



                # ----------------------------------------
                # Normal gap interpolation
                # ----------------------------------------

                interp_c=interp_contour(
                    c0,
                    c1,
                    t
                )


                corrected[f_fill][tid]={
                    "track_id":tid,
                    "centroid":(
                        cx_f,
                        cy_f
                    ),
                    "contour":interp_c,
                    "corrected":"gap_fill"
                }


                gaps_filled+=1



    print(
        f"  Gap filling: {gaps_filled} frames filled."
    )

    print(
        f"  Merge splits: {merges_fixed} merge events corrected."
    )


    return corrected, global_median_area

In [73]:
#@title Cell 8
# ============================================
# Cell 10
# Pass 2b helper functions
# ============================================


def last_known_before(tid, f_target, tracks):
    """
    Find a track's most recent real detection strictly before f_target.

    Returns:
        (frame, centroid)
    """

    hist = sorted(
        tracks[tid],
        key=lambda e: e["frame"]
    )


    before = [
        e for e in hist
        if e["frame"] < f_target
    ]


    if not before:
        return None, None


    e = before[-1]


    return (
        e["frame"],
        e["centroid"]
    )




def interp_centroid_for_track(tid, f_target, tracks):
    """
    Interpolate expected centroid of track tid at frame f_target.
    """

    hist = sorted(
        tracks[tid],
        key=lambda e:e["frame"]
    )


    before = [
        e for e in hist
        if e["frame"] <= f_target
    ]


    after = [
        e for e in hist
        if e["frame"] >= f_target
    ]



    if not before and not after:
        return None


    if not before:
        return after[0]["centroid"]


    if not after:
        return before[-1]["centroid"]



    eb = before[-1]
    ea = after[0]



    if eb["frame"] == ea["frame"]:

        return eb["centroid"]



    t = (
        f_target-eb["frame"]
    ) / (
        ea["frame"]-eb["frame"]
    )


    x = (
        eb["centroid"][0]*(1-t)
        +
        ea["centroid"][0]*t
    )


    y = (
        eb["centroid"][1]*(1-t)
        +
        ea["centroid"][1]*t
    )


    return (
        x,
        y
    )




def interp_contour_for_track(tid, f_target, tracks):
    """
    Interpolate expected contour of track tid at frame f_target.
    """

    hist = sorted(
        tracks[tid],
        key=lambda e:e["frame"]
    )


    before=[
        e for e in hist
        if e["frame"] <= f_target
        and e["contour"] is not None
    ]


    after=[
        e for e in hist
        if e["frame"] >= f_target
        and e["contour"] is not None
    ]



    if not before and not after:
        return None


    if not before:
        return after[0]["contour"]


    if not after:
        return before[-1]["contour"]



    eb=before[-1]
    ea=after[0]



    if eb["frame"] == ea["frame"]:

        return eb["contour"]



    t=(
        f_target-eb["frame"]
    ) / (
        ea["frame"]-eb["frame"]
    )


    return interp_contour(
        eb["contour"],
        ea["contour"],
        t
    )

In [74]:
#@title Cell 9
# ============================================
# Cell 11
# Pass 2b — n_cells enforcement
# Part 1/2
# ============================================


def enforce_n_cells(corrected, tracks, all_masks, n_frames,
                    height, width, n_cells,
                    global_median_area, peak_min_dist=15):
    """
    Enforce a known number of cells per frame.

    Handles:
        - under-detection:
            soft watershed recovery
            forced interpolation fallback

        - over-detection:
            remove least consistent detections
    """


    under_soft = 0
    under_hard = 0
    over_removed = 0



    track_lengths = {
        tid: len(hist)
        for tid, hist in tracks.items()
    }



    track_areas = {}

    for tid, hist in tracks.items():

        areas = [
            cv2.contourArea(e["contour"])
            for e in hist
            if e["contour"] is not None
        ]


        track_areas[tid] = (
            float(np.median(areas))
            if areas
            else global_median_area
        )



    for f in tqdm(
        range(n_frames),
        desc="Pass 2b: n_cells enforcement"
    ):


        present_ids = set(
            corrected[f].keys()
        )


        count = len(
            present_ids
        )


        if count == n_cells:
            continue



        all_tids = set(
            tracks.keys()
        )


        missing_ids = (
            all_tids -
            present_ids
        )



        # ======================================================
        # UNDER-DETECTION
        # ======================================================

        if count < n_cells and missing_ids:


            mask_f = all_masks[f]


            n_labels_soft, label_map_soft = (
                cv2.connectedComponents(mask_f)
            )



            # ------------------------------------------
            # Soft recovery:
            # split merged blob using watershed
            # ------------------------------------------

            for mid in list(missing_ids):


                if count >= n_cells:
                    break



                mid_prev_frame, mid_prev_xy = (
                    last_known_before(
                        mid,
                        f,
                        tracks
                    )
                )



                if mid_prev_xy is None:
                    continue



                best_cid = None
                best_dist = float("inf")



                for cid in present_ids:


                    if cid == mid:
                        continue



                    cand_xy = interp_centroid_for_track(
                        cid,
                        mid_prev_frame,
                        tracks
                    )



                    if cand_xy is None:
                        continue



                    d = np.linalg.norm(
                        np.array(cand_xy)
                        -
                        np.array(mid_prev_xy)
                    )



                    if d < best_dist:

                        best_dist = d
                        best_cid = cid



                if best_cid is None:
                    continue
                    _, cid_prev_xy = last_known_before(
                    best_cid,
                    f,
                    tracks
                )


                if cid_prev_xy is None:

                    cid_prev_xy = corrected[f][best_cid]["centroid"]



                # Locate neighbor blob

                ncx, ncy = corrected[f][best_cid]["centroid"]


                ix = int(
                    np.clip(
                        ncx,
                        0,
                        width-1
                    )
                )


                iy = int(
                    np.clip(
                        ncy,
                        0,
                        height-1
                    )
                )



                blob_id = label_map_soft[iy, ix]


                if blob_id == 0:
                    continue



                blob_mask = np.zeros(
                    (height,width),
                    dtype=np.uint8
                )


                blob_mask[
                    label_map_soft == blob_id
                ] = 255



                seed_xys = [
                    mid_prev_xy,
                    cid_prev_xy
                ]


                seed_tids = [
                    mid,
                    best_cid
                ]



                ws_markers,_ = watershed_seeded(
                    blob_mask,
                    seed_xys,
                    height,
                    width
                )



                recovered={}


                for s_idx,s_tid in enumerate(seed_tids):


                    region=np.zeros(
                        (height,width),
                        dtype=np.uint8
                    )


                    region[
                        ws_markers == s_idx+1
                    ] = 255



                    sub_contours,_=cv2.findContours(
                        region,
                        cv2.RETR_EXTERNAL,
                        cv2.CHAIN_APPROX_SIMPLE
                    )



                    if not sub_contours:
                        continue



                    sub_c=max(
                        sub_contours,
                        key=cv2.contourArea
                    )


                    sub_cen=contour_centroid(
                        sub_c
                    )


                    if sub_cen is None:
                        continue



                    recovered[s_tid]={
                        "centroid":sub_cen,
                        "contour":sub_c
                    }



                if mid not in recovered:
                    continue



                corrected[f][mid]={
                    "track_id":mid,
                    "centroid":recovered[mid]["centroid"],
                    "contour":recovered[mid]["contour"],
                    "corrected":"soft_recover"
                }



                if best_cid in recovered:

                    corrected[f][best_cid]={
                        "track_id":best_cid,
                        "centroid":recovered[best_cid]["centroid"],
                        "contour":recovered[best_cid]["contour"],
                        "corrected":corrected[f][best_cid].get(
                            "corrected"
                        )
                    }



                missing_ids.discard(mid)

                count += 1

                under_soft += 1




            # ------------------------------------------
            # Hard fallback interpolation
            # ------------------------------------------

            for mid in list(missing_ids):


                if count >= n_cells:
                    break



                pred_xy = interp_centroid_for_track(
                    mid,
                    f,
                    tracks
                )


                if pred_xy is None:
                    continue



                pred_ctr = interp_contour_for_track(
                    mid,
                    f,
                    tracks
                )


                corrected[f][mid]={
                    "track_id":mid,
                    "centroid":pred_xy,
                    "contour":pred_ctr,
                    "corrected":"forced"
                }



                count += 1

                under_hard += 1



        # ======================================================
        # OVER-DETECTION
        # ======================================================

        elif count > n_cells:


            excess = count - n_cells


            scores=[]



            for tid in present_ids:


                entry=corrected[f][tid]


                pred_xy=interp_centroid_for_track(
                    tid,
                    f,
                    tracks
                )


                det_xy=np.array(
                    entry["centroid"]
                )



                disp = (
                    np.linalg.norm(
                        np.array(pred_xy)-det_xy
                    )
                    if pred_xy is not None
                    else 999.0
                )



                med_area=track_areas.get(
                    tid,
                    global_median_area
                )


                area=(
                    cv2.contourArea(
                        entry["contour"]
                    )
                    if entry["contour"] is not None
                    else 0
                )


                area_err = (
                    abs(area-med_area)
                    /
                    max(med_area,1)
                )



                tlen = track_lengths.get(
                    tid,
                    1
                )



                score = (
                    disp
                    +
                    20.0*area_err
                    -
                    0.1*tlen
                )


                scores.append(
                    (
                        score,
                        tid
                    )
                )



            scores.sort(
                reverse=True
            )


            for _,tid in scores[:excess]:

                corrected[f].pop(
                    tid,
                    None
                )


                over_removed += 1



    return (
        under_soft,
        under_hard,
        over_removed
    )

In [75]:
#@title Cell 10
# ============================================
# Cell 12
# CSV export
# ============================================


def _serialize_contour(contour):
    """
    Flatten contour array into CSV string:
        x1:y1;x2:y2;...

    Returns empty string if contour is None.
    """

    if contour is None:
        return ""


    pts = contour.reshape(
        -1,
        2
    )


    return ";".join(
        f"{x}:{y}"
        for x,y in pts
    )




def export_tracks_csv(corrected, n_frames, csv_path):
    """
    Export final corrected detections.

    Columns:
        track_id
        frame
        center_x
        center_y
        contour_points
    """


    track_ids=set()


    for f in range(n_frames):

        track_ids.update(
            corrected[f].keys()
        )


    sorted_ids=sorted(
        track_ids
    )



    with open(
        csv_path,
        "w",
        newline=""
    ) as fh:


        writer=csv.writer(
            fh
        )


        writer.writerow(
            [
                "TRACK_ID",
                "FRAME",
                "POSITION_X",
                "POSITION_Y",
                "contour_points"
            ]
        )



        for tid in sorted_ids:


            for f in range(n_frames):


                entry=corrected[f].get(
                    tid
                )


                if entry is None:
                    continue



                cx,cy=entry["centroid"]


                contour_str=_serialize_contour(
                    entry.get("contour")
                )



                writer.writerow(
                    [
                        tid,
                        f,
                        cx,
                        cy,
                        contour_str
                    ]
                )


    print(
        f"Saved track CSV → {csv_path}"
    )

In [76]:
#@title Cell 11
# ============================================
# Cell 13
# Overlay drawing
# ============================================


def draw_overlay(frame_entries, height, width):
    """
    Draw corrected contours, centroids, and labels
    onto a transparent RGBA overlay frame.

    Returns:
        uint8 RGBA image
    """


    overlay = np.zeros(
        (height, width, 4),
        dtype=np.uint8
    )



    for entry in frame_entries:


        tid = entry["track_id"]

        cx, cy = entry["centroid"]

        contour = entry["contour"]

        kind = entry.get(
            "corrected",
            None
        )



        if contour is None:
            continue



        color = cell_color(tid) + (255,)



        pts = contour.reshape(
            -1,
            2
        )



        # -----------------------------------
        # Gap-filled contour
        # -----------------------------------

        if kind == "gap_fill":


            for i in range(
                0,
                len(pts),
                2
            ):


                p1 = tuple(
                    pts[i].astype(int)
                )


                p2 = tuple(
                    pts[
                        (i+1) % len(pts)
                    ].astype(int)
                )


                cv2.line(
                    overlay,
                    p1,
                    p2,
                    color,
                    1
                )



        # -----------------------------------
        # Merge split contour
        # -----------------------------------

        elif kind == "merge_split":


            for i in range(
                len(pts)
            ):


                p1 = tuple(
                    pts[i].astype(int)
                )


                p2 = tuple(
                    pts[
                        (i+1)%len(pts)
                    ].astype(int)
                )


                cv2.line(
                    overlay,
                    p1,
                    p2,
                    color,
                    3
                )



        # -----------------------------------
        # Soft recovery
        # -----------------------------------

        elif kind == "soft_recover":


            for i in range(
                0,
                len(pts),
                3
            ):


                p = tuple(
                    pts[i].astype(int)
                )


                cv2.circle(
                    overlay,
                    p,
                    1,
                    color,
                    -1
                )



        # -----------------------------------
        # Forced interpolation
        # -----------------------------------

        elif kind == "forced":


            cv2.drawMarker(
                overlay,
                (
                    int(cx),
                    int(cy)
                ),
                color,
                cv2.MARKER_CROSS,
                12,
                2
            )



        # -----------------------------------
        # Normal detection
        # -----------------------------------

        else:


            for i in range(
                len(pts)
            ):


                p1=tuple(
                    pts[i].astype(int)
                )


                p2=tuple(
                    pts[
                        (i+1)%len(pts)
                    ].astype(int)
                )


                cv2.line(
                    overlay,
                    p1,
                    p2,
                    color,
                    2
                )



        # centroid

        cv2.circle(
            overlay,
            (
                int(cx),
                int(cy)
            ),
            4,
            color,
            -1
        )



        cv2.putText(
            overlay,
            str(tid),
            (
                int(cx)+6,
                int(cy)-6
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            1,
            cv2.LINE_AA
        )



    return overlay

In [77]:
#@title Cell 12
# ============================================
# Cell 14
# Main pipeline
# Part 1/3
# ============================================


def detect_and_label(video_path,
                     overlay_video,
                     blur_ksize=5,
                     min_area=200,
                     peak_min_dist=15,
                     max_dist=50,
                     max_gap=10,
                     min_track_len=3,
                     merge_area_thresh=1.8,
                     n_cells=None,
                     roi_cx=None,
                     roi_cy=None,
                     roi_r=None,
                     csv_output=None,
                     threshold_method="otsu",
                     ):


    # -----------------------------------------
    # Load video metadata
    # -----------------------------------------

    cap = cv2.VideoCapture(
        video_path
    )


    if not cap.isOpened():

        raise FileNotFoundError(
            f"Cannot open video: {video_path}"
        )



    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )


    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )


    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )


    fps = (
        cap.get(
            cv2.CAP_PROP_FPS
        )
        or 30.0
    )



    print(
        f"Video: {width}x{height}, "
        f"{frame_count} frames @ {fps:.2f} fps"
    )



    # -----------------------------------------
    # Read frames
    # -----------------------------------------

    frames_gray=[]


    for _ in tqdm(
        range(frame_count),
        desc="Reading frames"
    ):


        ret, frame = cap.read()


        if not ret:
            break



        gray=cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )


        gray=cv2.normalize(
            gray,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        )


        frames_gray.append(
            gray.astype(np.uint8)
        )



    cap.release()



    frames_gray=np.array(
        frames_gray
    )


    background=np.median(
        frames_gray,
        axis=0
    ).astype(
        np.uint8
    )



    print(
        "Median background computed."
    )



    # -----------------------------------------
    # Circular ROI
    # -----------------------------------------

    _roi_cx = (
        roi_cx
        if roi_cx is not None
        else width/2
    )


    _roi_cy = (
        roi_cy
        if roi_cy is not None
        else height/2
    )



    if roi_r is not None:


        circle_mask = make_circle_mask(
            height,
            width,
            _roi_cx,
            _roi_cy,
            roi_r
        )


        print(
            f"ROI center=({_roi_cx:.1f},"
            f"{_roi_cy:.1f}), "
            f"radius={roi_r}"
        )


    else:

        circle_mask=None


        print(
            "No circular ROI."
        )
    # -----------------------------------------
    # Pass 1 — detection
    # -----------------------------------------

    kernel_size = int(width / 250)

    kernel_size += (kernel_size % 2 == 0)

    morph_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size,kernel_size)
    )


    ksize = (
        blur_ksize
        if blur_ksize % 2 == 1
        else blur_ksize + 1
    )


    all_detections=[]

    all_masks=[]



    for frame_idx in tqdm(
        range(len(frames_gray)),
        desc="Pass 1: detecting"
    ):


        gray = frames_gray[frame_idx]



        # Background subtraction

        fg=cv2.absdiff(
            gray,
            background
        )



        if circle_mask is not None:

            fg=cv2.bitwise_and(
                fg,
                fg,
                mask=circle_mask
            )



        # Blur

        fg_blur=cv2.GaussianBlur(
            fg,
            (ksize,ksize),
            0
        )



        # Otsu threshold

        mask = apply_threshold(fg_blur,method=threshold_method,bernsen_window=180,bernsen_contrast=25)



        # Morphological cleanup

        mask=cv2.morphologyEx(
            mask,
            cv2.MORPH_OPEN,
            morph_kernel
        )


        mask=cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            morph_kernel
        )



        all_masks.append(
            mask
        )



        contours_all,_=cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )



        detections=[]



        for contour in contours_all:


            if cv2.contourArea(contour) < min_area:

                continue



            cen=contour_centroid(
                contour
            )


            if cen is None:

                continue



            detections.append(
                {
                    "centroid":cen,
                    "contour":contour
                }
            )



        all_detections.append(
            detections
        )



    # -----------------------------------------
    # Track assignment
    # -----------------------------------------

    print(
        "Assigning track identities..."
    )


    tracks, frame_tracks = assign_tracks(
        all_detections,
        max_dist=max_dist
    )



    print(
        f"Raw tracks found: {len(tracks)}"
    )



    # -----------------------------------------
    # Correction sweep
    # -----------------------------------------

    print(
        "Running correction sweep..."
    )


    corrected, global_median_area = correction_sweep(
        tracks,
        frame_tracks,
        all_masks,
        n_frames=len(frames_gray),
        height=height,
        width=width,
        max_gap=max_gap,
        min_track_len=min_track_len,
        merge_area_thresh=merge_area_thresh
    )
        # -----------------------------------------
    # n_cells enforcement
    # -----------------------------------------

    if n_cells is not None:


        print(
            f"Enforcing n_cells={n_cells}"
        )


        u_soft, u_hard, o_rm = enforce_n_cells(
            corrected,
            tracks,
            all_masks,
            n_frames=len(frames_gray),
            height=height,
            width=width,
            n_cells=n_cells,
            global_median_area=global_median_area,
            peak_min_dist=peak_min_dist
        )


        print(
            f"Soft recoveries : {u_soft}"
        )

        print(
            f"Forced inserts  : {u_hard}"
        )

        print(
            f"Removed excess  : {o_rm}"
        )


    else:

        print(
            "n_cells not set - skipping enforcement."
        )



    # -----------------------------------------
    # CSV export
    # -----------------------------------------

    if csv_output is not None:

        export_tracks_csv(
            corrected,
            len(frames_gray),
            csv_output
        )



    corrected_frame_tracks=[

        list(corrected[f].values())

        for f in range(
            len(frames_gray)
        )

    ]



    # -----------------------------------------
    # Render overlay frames
    # -----------------------------------------

    temp_dir="overlay_frames_temp"


    os.makedirs(
        temp_dir,
        exist_ok=True
    )



    for frame_idx in tqdm(
        range(len(frames_gray)),
        desc="Rendering overlay"
    ):


        overlay_frame=draw_overlay(
            corrected_frame_tracks[frame_idx],
            height,
            width
        )



        if circle_mask is not None:


            cv2.circle(
                overlay_frame,
                (
                    int(_roi_cx),
                    int(_roi_cy)
                ),
                int(roi_r),
                (200,200,200,120),
                2
            )



        cv2.imwrite(
            os.path.join(
                temp_dir,
                f"frame_{frame_idx:05d}.png"
            ),
            overlay_frame
        )



    # -----------------------------------------
    # Composite overlay using ffmpeg
    # -----------------------------------------

    overlay_pattern=os.path.join(
        temp_dir,
        "frame_%05d.png"
    )



    cmd=[

        "ffmpeg",
        "-y",

        "-i",
        video_path,

        "-framerate",
        str(fps),

        "-i",
        overlay_pattern,

        "-filter_complex",
        "[1]format=rgba[ovr];[0][ovr]overlay",

        "-c:a",
        "copy",

        overlay_video
    ]



    subprocess.run(
        cmd,
        check=True
    )



    print(
        f"Saved overlay → {overlay_video}"
    )

In [ ]:
# ============================================
# Cell 13
# Notebook inputs + run
# ============================================


# -----------------------------
# Input / output files
# -----------------------------

VIDEO_PATH = "/content/C1_20260804_133401.mp4"          # change to your video filename
OVERLAY_PATH = "Uranus_cam_test-2.mp4"
CSV_PATH = "Uranus_cam_test-2.csv"

CAMERA = "URANUS"


# -----------------------------
# Detection parameters
# -----------------------------

BLUR_KSIZE = 5

if CAMERA == "URANUS":
    MIN_AREA = 100
elif CAMERA == "MARS":
    MIN_AREA = 0
else:
    raise ValueError(f"Unknown CAMERA: {CAMERA}")

PEAK_MIN_DIST = 15


# -----------------------------
# Tracking parameters
# -----------------------------

MAX_DIST = 75

MAX_GAP = 10

MIN_TRACK_LEN = 3


# -----------------------------
# Merge correction
# -----------------------------

MERGE_AREA_THRESH = 1.8



# -----------------------------
# Cell count enforcement
# -----------------------------
# Set to None to disable
# Example:
# N_CELLS = 3

N_CELLS = None



# -----------------------------
# Circular ROI
# -----------------------------
# Set ROI_R = None for full frame

if CAMERA == "URANUS":
    ROI_CX = 1910
    ROI_CY = 1104
    ROI_R = 1055
    THRESHOLD_METHOD = "bernsen"

elif CAMERA == "MARS":
    ROI_CX = 634
    ROI_CY = 483
    ROI_R = 440
    THRESHOLD_METHOD = "otsu"

else:
    raise ValueError(f"Unknown CAMERA: {CAMERA}")



# -----------------------------
# Run
# -----------------------------

detect_and_label(

    video_path=VIDEO_PATH,

    overlay_video=OVERLAY_PATH,

    blur_ksize=BLUR_KSIZE,

    min_area=MIN_AREA,

    peak_min_dist=PEAK_MIN_DIST,

    max_dist=MAX_DIST,

    max_gap=MAX_GAP,

    min_track_len=MIN_TRACK_LEN,

    merge_area_thresh=MERGE_AREA_THRESH,

    n_cells=N_CELLS,

    roi_cx=ROI_CX,

    roi_cy=ROI_CY,

    roi_r=ROI_R,

    csv_output=CSV_PATH,

    threshold_method=THRESHOLD_METHOD

)

Video: 3856x2180, 77 frames @ 7.64 fps


Reading frames: 100%|██████████| 77/77 [00:10<00:00,  7.55it/s]


Median background computed.
ROI center=(1910.0,1104.0), radius=1055


Pass 1: detecting: 100%|██████████| 77/77 [00:29<00:00,  2.61it/s]


Assigning track identities...
Raw tracks found: 32
Running correction sweep...
  Noise removal: dropped 2 short tracks.


Correction sweep: 100%|██████████| 30/30 [00:00<00:00, 53453.32it/s]


  Gap filling: 0 frames filled.
  Merge splits: 0 merge events corrected.
n_cells not set - skipping enforcement.
Saved track CSV → Uranus_cam_test-2.csv


Rendering overlay: 100%|██████████| 77/77 [00:13<00:00,  5.56it/s]
